# Retrieval-Augmented Generation Walkthrough

This notebook is the interactive route through the RAG chapter scaffold. It calls `rag_course_assistant.py` instead of duplicating retrieval and scoring logic, so the notebook, command-line examples, and repository checks stay aligned.

The default workflow uses a tiny sample corpus and dependency-light retrieval methods. Treat the first run as a plumbing check: confirm the corpus, chunking, retrieval settings, citation-based answer writer, and saved artifact names before replacing the sample files with a course corpus.

## Reader Preflight

Before running this notebook as public companion code:

- Start with the dependency check or smoke path. Real training, generation, or evaluation cells are usually guarded by flags such as `RUN_* = False`.
- Confirm dataset and model access, license or terms, and local paths before enabling external downloads or long runs.
- Keep secrets out of notebook cells. If a token is required, load it from the environment or `.env`, and keep `.env` secrets-only.
- Treat printed paths and saved JSON, CSV, and PNG artifacts as the evidence record. Rerun from a clean kernel before reporting results.

## 1. Preview The Sample Corpus

The first code cell locates the companion script, defines a small `run_script` helper, and calls `--preview-corpus`. Read the printed document IDs, chunk text, and question records before changing retrieval settings; otherwise a metric change can be caused by data drift rather than the retrieval method.

In [ ]:
from pathlib import Path
import subprocess
import sys


def find_chapter_dir() -> Path:
    script_name = "rag_course_assistant.py"
    chapter_name = "chapter_retrieval_augmented_generation"
    for base in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (base, base / chapter_name, base / "code" / chapter_name):
            if (candidate / script_name).exists():
                return candidate.resolve()
    raise FileNotFoundError(f"Could not locate {script_name}")


CHAPTER_DIR = find_chapter_dir()
SCRIPT = CHAPTER_DIR / "rag_course_assistant.py"


def run_script(*args: str) -> None:
    subprocess.run([sys.executable, str(SCRIPT), *args], cwd=CHAPTER_DIR, check=True)


run_script("--preview-corpus")

## 2. Run The Baseline And Save Artifacts

This baseline run keeps the default chunking, lexical retrieval, deterministic dense vectors, and citation-based answer writer. The command writes `chunks.jsonl`, `retrieval_results.jsonl`, `answers.jsonl`, and `run_summary.json` under `artifacts/walkthrough/`; those files are the evidence to inspect or cite in the homework report.

In [ ]:
run_script("--run", "--save-artifacts", "--artifact-dir", "artifacts/walkthrough")

## 3. Interpret The Baseline Artifacts

`chunks.jsonl` is the data-shaping record: check document IDs, chunk IDs, section labels, excerpts, and chunk counts before trusting any retrieval score. `retrieval_results.jsonl` is the retrieval diagnostic file; for each question, inspect the top-ranked chunk IDs, ranks, scores, latency, and whether the gold evidence was marked acceptable. `answers.jsonl` is the answer-side record; compare the prompt-only, RAG baseline, and controlled rows for answer score, cited chunk IDs, citation faithfulness, and generation latency.

Use `run_summary.json` for the compact comparison table, but do not rely on the summary alone. A public report should explain at least one retrieved example where the method helped, one miss where the right source was absent or ranked too low, and whether the answer writer used only chunks that were actually in the retrieved context.

## 4. Try One Controlled Retrieval Change

The comparison changes chunk size, overlap, top-k, and retrieval method while keeping the question set and scoring rubric fixed. Use one controlled change at a time when preparing a report, and keep generated artifacts in an ignored `artifacts/` directory unless they have been reviewed for public release.

In [ ]:
run_script(
    "--run",
    "--chunk-tokens",
    "60",
    "--overlap-tokens",
    "15",
    "--top-k",
    "4",
    "--rag-method",
    "dense",
    "--controlled-method",
    "hybrid",
    "--save-artifacts",
    "--artifact-dir",
    "artifacts/chunk60_top4",
)

## 5. What To Report

For a reportable RAG comparison, name the corpus version, question set, chunk size, overlap, top-k value, retrieval method, answer writer, and artifact directory. Report retrieval metrics and answer metrics together: better retrieval is only useful if the answer uses the retrieved evidence faithfully, and a fluent answer without matching citations should be treated as unsupported.

When moving beyond the sample corpus, keep course documents, prompts, question JSONL, retrieval outputs, and scores as ordinary data files. Store only secrets such as API keys in `.env`, and review generated artifacts before committing anything derived from private, licensed, or student-submitted material.

## Reader Report Checklist

Before using results from this notebook in a report or downstream example, record:

- The notebook name, companion script, package versions, device, seed, and any guarded flags you enabled.
- The dataset or sample-data source, license or terms, split policy, and any preprocessing or synthetic fallback used.
- The exact artifact files that support the result, preferably saved JSON, CSV, or PNG files rather than transient cell output.
- The validation evidence used for model or configuration selection, and whether any final-test cell was run exactly once.
- The main limitation of the run, such as tiny synthetic data, missing optional dependencies, short training budget, or unavailable model access.